<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/UDP_text_reciever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 【受信側】文字を UDP で受け取る 📥

このノートブックは **受け取る人** が動かします。
インターネットからの公開アドレスを作り、そこに届いた文字を表示します。

送る人は **別の Colab**（別ノート「【送信側】」）で動かします。
あなたがここで表示する **公開アドレス** を、送る人に伝えてください。

実行する順番：**① 受け取りの準備 → ② 公開アドレスを取得 → ③ 受信を監視**

> UDP は「送りっぱなし」の通信です。送る人が送っても、ここに届かないことがあります。それが正常です。

## ① 受け取りの準備

届いた文字は **受信箱（INBOX）** にためていきます。
このセルは1回だけでOK。もう一度実行しても大丈夫です（前の受け取りを閉じてから開き直します）。

In [1]:
import socket, threading, datetime

PORT = 50007
try:
    rx.close()      # 前の受け取りが残っていれば閉じる
except Exception:
    pass

rx = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)   # UDP のソケット
rx.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
rx.bind(("0.0.0.0", PORT))

INBOX = []          # 届いた文字をためる受信箱
_stop = threading.Event()

def receiver():
    while not _stop.is_set():
        try:
            data, addr = rx.recvfrom(65535)              # 届くのを待つ
        except OSError:
            break
        text = data.decode("utf-8", errors="replace")    # バイト列を文字に戻す
        ts = datetime.datetime.now().strftime("%H:%M:%S")
        INBOX.append((ts, text, addr))                    # 受信箱に入れるだけ

threading.Thread(target=receiver, daemon=True).start()
print(f"受け取り準備OK（ポート {PORT}）")


受け取り準備OK（ポート 50007）


## ② 公開アドレスを取得

インターネットから届けてもらうための住所（公開アドレス）を作ります。
最初の1回だけインストールが走るので、少し時間がかかります。

実行すると **`udp://……`** という公開アドレスが表示されます。
**これをそのまま送る人に伝えてください**（送信側ノートに貼り付けてもらいます）。

※ 無料のため、しばらく（約60分）でアドレスが切れます。切れたらこの②をもう一度実行し、新しいアドレスを送る人に伝え直してください。

In [2]:
!pip install -q pinggy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 21.2 MB/s eta 0:00:00


In [3]:
import pinggy, re, time

tunnel = pinggy.start_udptunnel(forwardto=PORT)   # localhost:50007 に届ける公開アドレスを作る

PUBLIC_URL = None
for _ in range(20):
    urls = tunnel.urls or []
    if urls:
        PUBLIC_URL = urls[0]
        break
    time.sleep(1)

print("=" * 48)
if PUBLIC_URL:
    print("送る人に伝える公開アドレス：")
    print("   ", PUBLIC_URL)
else:
    print("アドレスをまだ取得できていません。もう一度このセルを実行してください。")
print("=" * 48)


送る人に伝える公開アドレス：
    udp://bwhxu-136-119-70-15.run.pinggy-free.link:29570


## ③ 受信を監視

このセルを実行すると、届いた文字を**リアルタイムで表示**し続けます。
送る人に送ってもらってください。届くたびに下に1行ずつ増えます。

**止めるとき**：セル左の停止ボタン（■）を押すか、Ctrl+C。

In [7]:
import time

print("受信を監視中です。送る人に送ってもらってください。")
print("（止めるときは停止ボタン ■ ）\n")

seen = 0
try:
    while True:
        if len(INBOX) > seen:
            for ts, text, addr in INBOX[seen:]:
                print(f"[{ts}] {addr[0]}:{addr[1]} 「{text}」")
            seen = len(INBOX)
        time.sleep(0.3)
except KeyboardInterrupt:
    print(f"\n監視を止めました（受信箱：{len(INBOX)} 通）")


受信を監視中です。送る人に送ってもらってください。
（止めるときは停止ボタン ■ ）

[03:52:03] 127.0.0.1:33047 「こんにちは UDP」
[03:54:12] 127.0.0.1:57952 「こんにちは UDP」
[03:54:40] 127.0.0.1:57952 「test-0」
[03:54:41] 127.0.0.1:57952 「test-1」
[03:54:41] 127.0.0.1:57952 「test-2」
[03:54:41] 127.0.0.1:57952 「test-3」
[03:54:42] 127.0.0.1:57952 「test-4」
[03:54:42] 127.0.0.1:57952 「test-5」
[03:54:42] 127.0.0.1:57952 「test-6」
[03:54:42] 127.0.0.1:57952 「test-7」
[03:54:43] 127.0.0.1:57952 「test-8」
[03:54:43] 127.0.0.1:57952 「test-9」
[03:55:34] 127.0.0.1:39076 「ローカル確認」
[03:55:34] 127.0.0.1:54439 「トンネル確認」

監視を止めました（受信箱：14 通）


## 受信箱の中身を見る（いつでも）

監視を止めたあとや、まとめて確認したいときに実行します。

In [8]:
print(f"受信箱：{len(INBOX)} 通")
for ts, text, addr in INBOX[-30:]:
    print(f"  [{ts}] {addr[0]}:{addr[1]} 「{text}」")


受信箱：14 通
  [03:52:03] 127.0.0.1:33047 「こんにちは UDP」
  [03:54:12] 127.0.0.1:57952 「こんにちは UDP」
  [03:54:40] 127.0.0.1:57952 「test-0」
  [03:54:41] 127.0.0.1:57952 「test-1」
  [03:54:41] 127.0.0.1:57952 「test-2」
  [03:54:41] 127.0.0.1:57952 「test-3」
  [03:54:42] 127.0.0.1:57952 「test-4」
  [03:54:42] 127.0.0.1:57952 「test-5」
  [03:54:42] 127.0.0.1:57952 「test-6」
  [03:54:42] 127.0.0.1:57952 「test-7」
  [03:54:43] 127.0.0.1:57952 「test-8」
  [03:54:43] 127.0.0.1:57952 「test-9」
  [03:55:34] 127.0.0.1:39076 「ローカル確認」
  [03:55:34] 127.0.0.1:54439 「トンネル確認」


## （任意）自分でもテスト送信してみる

送る人を待たずに、受信側だけで動作確認したいとき用です。
- `via="local"` … 自分の中だけ（127.0.0.1）。ほぼ確実に届く。
- `via="tunnel"` … 公開アドレス経由。インターネットを一周して戻る。消えることがある。

In [9]:
import socket, re, time

_tx = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

def self_test(msg="テスト", via="tunnel", wait=3.0):
    if via == "tunnel":
        m = re.search(r"([\w.\-]+):(\d+)", (PUBLIC_URL or "").replace("udp://", ""))
        if not m:
            print("公開アドレスがありません。先に②を実行してください。")
            return
        target = (m.group(1), int(m.group(2)))
    else:
        target = ("127.0.0.1", PORT)

    before = len(INBOX)
    _tx.sendto(msg.encode("utf-8"), target)
    print(f"送信（{via}）: 「{msg}」")
    t0 = time.time()
    while time.time() - t0 < wait and len(INBOX) == before:
        time.sleep(0.1)
    new = INBOX[before:]
    if new:
        for ts, text, addr in new:
            print(f"  ↩ 戻ってきた [{ts}] 「{text}」")
    else:
        print("  … 今回は戻ってきませんでした（UDP なので消えることがあります）")

self_test("ローカル確認", via="local")
self_test("トンネル確認", via="tunnel")


送信（local）: 「ローカル確認」
  ↩ 戻ってきた [03:55:47] 「ローカル確認」
送信（tunnel）: 「トンネル確認」
  ↩ 戻ってきた [03:55:47] 「トンネル確認」


## かたづけ

終わったら公開アドレスを閉じます。

In [10]:
tunnel.stop()
_stop.set()
rx.close()
print("おわり")


おわり


### まとめ（受信側）

- 公開アドレス `udp://……` を作り、送る人に伝えた。
- 届いた文字は受信箱にたまり、監視セルで表示した。
- 送られても届かないことがある。再送はされない。これが UDP。